# LIAR2 Label-Granularity Experiments — 6 → 3 → 2 Class Collapse

**Purpose.** Re-run the *identical* chronological, restricted-metadata DA-MiniLM-RWAC
protocol on LIAR2 with the 6 ordinal veracity classes collapsed into **3 classes**
`{false, mixed, true}` and into **2 classes** `{false, true}`, then compare
DA-MiniLM-RWAC against MiniLM-LogReg-RW and Uniform-LR at each granularity.

**Research question.** Does DA-MiniLM-RWAC's ~0.031 macro-F1 advantage over
MiniLM-LogReg-RW survive label coarsening, or is it an artifact of the hard 6-way
minority classes? A *vanishing* advantage under collapse is itself a reportable
result. The 3-class `{false, mixed, true}` taxonomy is also the *same* one MultiFC
is collapsed to, making these LIAR2 numbers directly comparable to the
external-replication set.

**Methodological invariants (unchanged from the main pipeline).**

1. **Collapse happens *after* the chronological split / dedup / leakage-drop / time_rank
   pipeline.** Only the `label_id` column is remapped, so the 3-/2-class runs sit on
   the *exact same rows* as the 6-class run — a clean, paired comparison.
2. **Macro-F1 stays the primary metric.** Accuracy inflation from the enlarged
   majority bin is *worse* under collapse, not better, so macro-F1 is even more
   necessary here.
3. **Class-balance sample weights are recomputed per granularity** (fresh model +
   runner with the correct `num_classes`); the 6-class weights are never reused.
4. **Embeddings are computed once and reused across all granularities** — collapsing
   labels does not change the model inputs, so re-encoding would be wasted compute.
5. **The 2-class boundary places `half-true` on the *true* side** (LIAR convention).
   This is a documented judgment call; a robustness variant (`half-true → false`) is
   run and reported alongside it.

## How to run on Kaggle

1. Enable **Internet** (Settings → Internet → On) — required for MiniLM embeddings.
2. Enable a **GPU** accelerator (strongly recommended for the MLP).
3. **Paste each of your project's class files into the marked cell below**, in the
   listed order, *or* attach them as a utility-script/dataset and import them. This
   notebook does not re-implement any model/preprocessing logic — it imports it and
   only adds the label-collapse layer on top.
4. Update the three LIAR2 CSV paths in **Section 0**.


## Section 0 — Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd

# ----- EDIT THESE THREE PATHS to match your Kaggle LIAR2 dataset -----
LIAR2_TRAIN = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/train.csv"
LIAR2_VALID = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/valid.csv"
LIAR2_TEST  = "/kaggle/input/datasets/badalahmmed/liar-dataset/liar2/liar2/test.csv"

OUTPUT_DIR = "/kaggle/working/label_collapse_results"

# ----- Experiment configuration -----
# Restricted-metadata text mode is the paper's headline condition.
# (In-code the column is named "claim_plus_safe_metadata"; the paper calls it
#  "restricted metadata" — same thing.)
TEXT_MODE   = "claim_plus_safe_metadata"
SPLIT_MODE  = "chronological"          # the paper's headline split
SEEDS       = [0, 1, 2, 3, 4]          # 5-seed battery
N_BOOT      = 2000                     # paired-bootstrap resamples

# Which collapse schemes to run. "6class" is included as the reference point.
SCHEMES = ["6class", "3class", "2class", "2class_halftrue_false"]

# Set True to also run the plain claim-only text mode at each granularity.
ALSO_RUN_CLAIM_ONLY = False

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config loaded. Output dir:", OUTPUT_DIR)

## Section 1 — Project class definitions

Paste the following class files **into the cell below**, in this order (top to bottom),
exactly as they exist in your project — or attach them as an importable utility script
and replace the cell with the corresponding `import` statements:

```
liar2_preprocessor.py          ->  class Liar2Preprocessor
temporal_splitter.py           ->  class TemporalSplitter
da_minilm_model.py             ->  class DAMiniLMModel  (+ set_all_seeds, helpers)
da_minilm_experiment_runner.py ->  class DAMiniLMExperimentRunner (+ all module-level helpers)
```

These are the same four files `run_all.py` imports. This notebook adds **no** modelling
or preprocessing logic of its own; it only remaps the label column and drives the
existing runner. After pasting, the classes `Liar2Preprocessor`, `TemporalSplitter`,
`DAMiniLMModel`, and `DAMiniLMExperimentRunner` must all be defined in the namespace.

In [ ]:
"""
liar2_preprocessor.py

One class, LIAR2 only. Every preprocessing step is its own method so you
can call them individually and inspect intermediate output, instead of
only seeing the final combined result.

Two distinct splits are tracked, on purpose, because they answer different
questions:
    - `official_split` : which of train.csv/valid.csv/test.csv a row came
                          from. This is LIAR2's own distributed split
                          (~80/10/10). Useful for reporting results against
                          the split other papers on LIAR2 use.
    - chronological_split() / random_stratified_split() : the project's
      OWN 70/15/15 re-split of the full recombined, chronologically-sorted
      dataset, matching liar2-split-and-input-ablation-experiments*.ipynb
      and da-minilm-main-model-v1-0.ipynb. This is the split your reported
      DA-MiniLM-RWAC / MiniLM-LogReg-RW numbers were actually evaluated on.

These are NOT the same partition (verified: the official split is
18369/2297/2296 = ~80/10/10; the canonical re-split ratios are 70/15/15
over the recombined data). Don't mix them up when reporting results.

Usage
-----
    from liar2_preprocessor import Liar2Preprocessor

    p = Liar2Preprocessor()

    # step by step, so you can inspect each stage:
    df = p.load_raw(train_path, valid_path, test_path)
    df = p.clean_statement_text(df)
    df = p.normalize_labels(df)
    df = p.filter_invalid_rows(df)
    df = p.parse_and_sort_by_date(df)
    df = p.build_text_modes(df)
    df = p.drop_leakage_columns(df)
    df = p.add_time_rank(df)

    # or all at once:
    df = p.run_pipeline(train_path, valid_path, test_path)

    # re-splits, computed on the fully-cleaned df:
    train_idx, valid_idx, test_idx = p.chronological_split(df)
    train_idx, valid_idx, test_idx = p.random_stratified_split(df, seed=42)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


class Liar2Preprocessor:

    LABEL_ORDER = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
    NUMERIC_LABEL_MAP = {0: "pants-fire", 1: "false", 2: "barely-true",
                         3: "half-true", 4: "mostly-true", 5: "true"}
    # Included in model text (restricted metadata, not leakage-risk):
    SAFE_METADATA_COLUMNS = ["subject", "speaker", "speaker_description", "state_info", "context"]
    # Dropped entirely -- justification and speaker credibility-history counts are leakage risks:
    EXCLUDED_COLUMNS = ["justification", "true_counts", "mostly_true_counts", "half_true_counts",
                        "mostly_false_counts", "false_counts", "pants_on_fire_counts"]

    def __init__(
        self,
        train_ratio: float = 0.70,
        valid_ratio: float = 0.15,
        test_ratio: float = 0.15,
        date_coverage_threshold: float = 0.80,
        seed: int = 42,
    ):
        assert abs((train_ratio + valid_ratio + test_ratio) - 1.0) < 1e-9, \
            "train_ratio + valid_ratio + test_ratio must sum to 1.0"
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio
        self.date_coverage_threshold = date_coverage_threshold
        self.seed = seed
        self.label_to_id = {label: i for i, label in enumerate(self.LABEL_ORDER)}

    # ================================================================
    # Step 1: Load, tagging each row with its OFFICIAL file-based split
    # ================================================================

    def load_raw(self, train_path, valid_path, test_path) -> pd.DataFrame:
        """Load the three official LIAR2 CSVs and concatenate them, tagging
        each row with `official_split` = "train"/"valid"/"test" based on
        which file it came from. Nothing else is cleaned yet -- this is
        the raw concatenation, useful for auditing raw label distributions
        or row counts per official split before any cleaning happens.
        """
        frames = []
        for path, split_name in [(train_path, "train"), (valid_path, "valid"), (test_path, "test")]:
            part = pd.read_csv(path)
            part["official_split"] = split_name
            part["source_file"] = Path(path).name
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)

        print("Loaded raw LIAR2 rows by official_split:")
        print(df["official_split"].value_counts().reindex(["train", "valid", "test"]))
        return df

    # ================================================================
    # Step 2: Text cleaning
    # ================================================================

    @staticmethod
    def clean_text(x) -> str:
        if pd.isna(x):
            return ""
        return re.sub(r"\s+", " ", str(x)).strip()

    def clean_statement_text(self, df: pd.DataFrame) -> pd.DataFrame:
        """Whitespace-collapse and strip the `statement` column."""
        df = df.copy()
        df["statement"] = df["statement"].apply(self.clean_text)
        return df

    # ================================================================
    # Step 3: Label normalization
    # ================================================================

    def _normalize_single_label(self, x):
        if pd.isna(x):
            return np.nan
        try:
            if str(x).strip().replace(".0", "").isdigit():
                xi = int(float(x))
                if xi in self.NUMERIC_LABEL_MAP:
                    return self.NUMERIC_LABEL_MAP[xi]
        except Exception:
            pass
        x = str(x).strip().lower().replace("_", "-")
        x = re.sub(r"\s+", "-", x)
        mapping = {
            "pants-fire": "pants-fire", "pants-on-fire": "pants-fire", "pantsonfire": "pants-fire",
            "false": "false",
            "barely-true": "barely-true", "barelytrue": "barely-true",
            "half-true": "half-true", "halftrue": "half-true",
            "mostly-true": "mostly-true", "mostlytrue": "mostly-true",
            "true": "true",
        }
        return mapping.get(x, x)

    def normalize_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """Map raw label (numeric 0-5 or text, with spelling variants) to
        the canonical 6-class label string. Invalid/unmapped labels are
        left as-is here -- use filter_invalid_rows() to drop them."""
        df = df.copy()
        df["label"] = df["label"].apply(self._normalize_single_label)
        return df

    # ================================================================
    # Step 4: Drop empty statements / unmapped labels, assign label_id
    # ================================================================

    def filter_invalid_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drop rows with an empty statement or a label outside LABEL_ORDER,
        then assign the integer `label_id`. Prints how many rows were
        removed at each stage so you can see exactly what got dropped."""
        n_start = len(df)

        df = df[df["statement"].str.len() > 0].copy()
        n_after_empty = len(df)

        df = df[df["label"].isin(self.LABEL_ORDER)].copy()
        n_after_label = len(df)

        df["label_id"] = df["label"].map(self.label_to_id).astype(int)

        print(f"filter_invalid_rows: {n_start} -> {n_after_empty} "
              f"(dropped {n_start - n_after_empty} empty statements) -> "
              f"{n_after_label} (dropped {n_after_empty - n_after_label} unmappable labels)")
        return df.reset_index(drop=True)

    # ================================================================
    # Step 5: Date parsing + chronological sort (with coverage fallback)
    # ================================================================

    def parse_and_sort_by_date(self, df: pd.DataFrame) -> pd.DataFrame:
        """Parse `date` into `date_parsed`. If more than
        `date_coverage_threshold` fraction of rows have a valid date,
        sort the whole frame chronologically; otherwise fall back to the
        original row order (matches the canonical notebooks' behavior).
        Assigns `claim_id` as the post-sort row index either way.
        Sets `df.attrs["sort_source"]` and `df.attrs["date_coverage"]`.
        """
        df = df.copy()
        df["date_parsed"] = pd.to_datetime(df["date"], errors="coerce") if "date" in df.columns else pd.NaT

        n_total = len(df)
        n_dated = int(df["date_parsed"].notna().sum())
        coverage = n_dated / n_total if n_total else 0.0

        if coverage > self.date_coverage_threshold:
            df = df.sort_values("date_parsed").reset_index(drop=True)
            sort_source = "date"
        else:
            df = df.reset_index(drop=True)
            sort_source = "row_order"

        df["claim_id"] = np.arange(len(df))
        df.attrs["sort_source"] = sort_source
        df.attrs["date_coverage"] = coverage

        print(f"parse_and_sort_by_date: {n_dated}/{n_total} rows have a valid date "
              f"({coverage:.2%}) -> sort_source = '{sort_source}'")
        return df

    # ================================================================
    # Step 6: Build both text-input modes (for the claim-only vs
    # claim+metadata ablation)
    # ================================================================

    def _build_claim_only(self, row) -> str:
        return f"claim: {row['statement']}"

    def _build_claim_plus_safe_metadata(self, row) -> str:
        parts = [f"claim: {row['statement']}"]
        for col in self.SAFE_METADATA_COLUMNS:
            if col in row.index:
                val = self.clean_text(row.get(col, ""))
                if val:
                    parts.append(f"{col}: {val}")
        return " [SEP] ".join(parts)

    def build_text_modes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adds two columns:
            model_text__claim_only
            model_text__claim_plus_safe_metadata
        so both conditions of the text-input ablation are available at
        once without re-running preprocessing."""
        df = df.copy()
        df["model_text__claim_only"] = df.apply(self._build_claim_only, axis=1)
        df["model_text__claim_plus_safe_metadata"] = df.apply(self._build_claim_plus_safe_metadata, axis=1)
        return df

    # ================================================================
    # Step 7: Drop leakage-risk columns entirely
    # ================================================================

    def drop_leakage_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Drops `justification` and all speaker credibility-history count
        columns entirely (not just excludes them from model text), so they
        can't be accidentally used downstream."""
        present = [c for c in self.EXCLUDED_COLUMNS if c in df.columns]
        df = df.drop(columns=present)
        print(f"drop_leakage_columns: removed {present if present else '(none present)'}")
        return df

    # ================================================================
    # Duplicate / conflicting-label handling (opt-in, NOT called by
    # run_pipeline automatically -- see note below)
    # ================================================================

    @staticmethod
    def _normalize_for_overlap(x: str) -> str:
        """Aggressive normalization used ONLY for duplicate detection,
        never for the text fed to a model."""
        x = str(x).lower()
        x = re.sub(r"[^a-z0-9\s]", "", x)
        return re.sub(r"\s+", " ", x).strip()

    def audit_duplicate_statements(self, df: pd.DataFrame) -> pd.DataFrame:
        """Diagnostic only -- does NOT modify df. Groups rows by normalized
        statement text and reports, per group: row count, distinct labels
        present, and whether the group has a genuine label conflict (same
        claim, different verdicts)."""
        work = df.copy()
        work["_norm"] = work["statement"].apply(self._normalize_for_overlap)

        grouped = work.groupby("_norm").agg(
            n_rows=("label", "size"),
            distinct_labels=("label", lambda s: sorted(set(s))),
        ).reset_index()
        grouped["is_duplicate_group"] = grouped["n_rows"] > 1
        grouped["has_label_conflict"] = grouped["distinct_labels"].apply(len) > 1
        return grouped

    def identify_conflicting_label_groups(self, audit_df: pd.DataFrame) -> pd.DataFrame:
        """Subset of audit_duplicate_statements() output where the same
        normalized claim carries more than one distinct label."""
        return audit_df[audit_df["has_label_conflict"]].copy()

    def drop_duplicate_statements(
        self,
        df: pd.DataFrame,
        policy: str = "keep_first",
        audit_df=None,
    ) -> pd.DataFrame:
        """
        NOT part of the canonical pipeline -- the source notebooks this
        project is based on (liar2-split-and-input-ablation-experiments*
        .ipynb, da-minilm-main-model-v1-0.ipynb) contain no deduplication
        step, so calling this will change your row count relative to any
        previously reported results. Call it deliberately, not by default.

        policy:
            "keep_first"              -> for every duplicate group (exact
                                          or conflicting-label), keep only
                                          the chronologically earliest row.
            "drop_conflicting_groups" -> conflicting-label groups are
                                          DROPPED ENTIRELY (all rows in the
                                          group removed); non-conflicting
                                          duplicate groups still keep the
                                          earliest row.
        """
        if audit_df is None:
            audit_df = self.audit_duplicate_statements(df)

        work = df.copy()
        work["_norm"] = work["statement"].apply(self._normalize_for_overlap)

        conflicts = self.identify_conflicting_label_groups(audit_df)
        n_conflict_groups = len(conflicts)
        n_conflict_rows = int(work["_norm"].isin(conflicts["_norm"]).sum())
        if n_conflict_groups > 0:
            print(f"drop_duplicate_statements: {n_conflict_groups} conflicting-label group(s) "
                  f"covering {n_conflict_rows} row(s) found.")

        if policy == "drop_conflicting_groups" and n_conflict_groups > 0:
            work = work[~work["_norm"].isin(conflicts["_norm"])].copy()

        before = len(work)
        sort_col = "date_parsed" if "date_parsed" in work.columns else "claim_id"
        work = work.sort_values(sort_col).drop_duplicates(subset=["_norm"], keep="first")
        after = len(work)
        print(f"drop_duplicate_statements ({policy}): {before} -> {after} (removed {before - after})")

        return work.drop(columns=["_norm"]).reset_index(drop=True)

    # ================================================================
    # Step 8: Time features (for Time2Vec / domain-adversarial component)
    # ================================================================

    def add_time_rank(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adds `time_rank` in [0, 1], a global linspace over the full
        chronologically-sorted frame. Must be called AFTER
        parse_and_sort_by_date(). Matches
        liar2-split-and-input-ablation-experiments3.ipynb and the main
        model notebook exactly (global rank, computed before any re-split)."""
        df = df.copy()
        df["time_rank"] = np.linspace(0.0, 1.0, len(df))
        return df

    def make_domain_buckets(self, time_rank_train: np.ndarray, num_domains: int = 5) -> np.ndarray:
        """Quantile-bucket TRAIN rows' time_rank into `num_domains` discrete
        ids, for domain-adversarial training. Pass only train-split
        time_rank values -- never valid/test -- to avoid leakage into the
        domain-bucket boundaries."""
        quantiles = np.quantile(time_rank_train, np.linspace(0, 1, num_domains + 1))
        quantiles[0] = -np.inf
        quantiles[-1] = np.inf
        return np.digitize(time_rank_train, quantiles[1:-1], right=True)

    # ================================================================
    # Re-splits (project's own 70/15/15 protocol, distinct from
    # official_split)
    # ================================================================

    def chronological_split(self, df: pd.DataFrame):
        """Contiguous 70/15/15 split in row order. Requires df to already
        be chronologically sorted (i.e. call parse_and_sort_by_date() first
        with sort_source == 'date')."""
        n = len(df)
        train_end = int(n * self.train_ratio)
        valid_end = int(n * (self.train_ratio + self.valid_ratio))
        return (
            np.arange(0, train_end),
            np.arange(train_end, valid_end),
            np.arange(valid_end, n),
        )

    def random_stratified_split(self, df: pd.DataFrame, seed: int = None):
        """70/15/15 stratified-by-label split, re-sorted chronologically
        within each split so recency weighting remains meaningful."""
        seed = self.seed if seed is None else seed
        all_idx = np.arange(len(df))
        y = df["label_id"].values

        train_idx, temp_idx = train_test_split(
            all_idx, train_size=self.train_ratio, random_state=seed, stratify=y,
        )
        y_temp = y[temp_idx]
        valid_relative_ratio = self.valid_ratio / (self.valid_ratio + self.test_ratio)
        valid_idx, test_idx = train_test_split(
            temp_idx, train_size=valid_relative_ratio, random_state=seed, stratify=y_temp,
        )
        return np.sort(train_idx), np.sort(valid_idx), np.sort(test_idx)

    def get_split(self, df: pd.DataFrame, split_mode: str, seed: int = None):
        """Added so Liar2Preprocessor is genuinely interchangeable with
        TemporalSplitter wherever a `.splitter` is expected (e.g.
        DAMiniLMExperimentRunner.run_full_battery()) -- confirmed missing
        via a real crash (AttributeError) when first wiring the two
        together in a full end-to-end run; not just a hypothetical gap."""
        if split_mode == "chronological":
            return self.chronological_split(df)
        if split_mode == "random_stratified":
            return self.random_stratified_split(df, seed=seed)
        raise ValueError(f"Unknown split_mode: {split_mode}")

    def make_rolling_windows(
        self,
        df: pd.DataFrame,
        min_train_frac: float = 0.50,
        valid_frac: float = 0.10,
        test_frac: float = 0.10,
        step_frac: float = 0.10,
    ) -> list:
        """Expanding-window temporal evaluation windows, for the temporal-
        generalization / rolling-window diagnostics."""
        n = len(df)
        windows = []
        train_end = int(n * min_train_frac)
        valid_size = int(n * valid_frac)
        test_size = int(n * test_frac)
        step = max(1, int(n * step_frac))

        window_id = 0
        while train_end + valid_size + test_size <= n:
            valid_end = train_end + valid_size
            test_end = valid_end + test_size
            train_idx = np.arange(0, train_end)
            valid_idx = np.arange(train_end, valid_end)
            test_idx = np.arange(valid_end, test_end)

            windows.append({
                "window_id": window_id,
                "train_idx": train_idx, "valid_idx": valid_idx, "test_idx": test_idx,
                # These three were missing until a real end-to-end dry run
                # crashed inside DAMiniLMExperimentRunner.run_rolling_window_evaluation(),
                # which expects the same keys TemporalSplitter.make_rolling_windows()
                # returns -- added here so the two are genuinely interchangeable,
                # not just interchangeable for chronological_split/random_stratified_split.
                "train_end_frac": train_end / n, "valid_end_frac": valid_end / n, "test_end_frac": test_end / n,
                "n_train": len(train_idx), "n_valid": len(valid_idx), "n_test": len(test_idx),
            })
            train_end += step
            window_id += 1
        return windows

    # ================================================================
    # Full pipeline (runs every step above, in order)
    # ================================================================

    def run_pipeline(self, train_path, valid_path, test_path) -> pd.DataFrame:
        """Runs every step in sequence:
            load_raw -> clean_statement_text -> normalize_labels ->
            filter_invalid_rows -> parse_and_sort_by_date ->
            build_text_modes -> drop_leakage_columns -> add_time_rank
        Each step prints its own diagnostic line, so a single call here
        gives you the full audit trail, not just the final DataFrame.
        """
        df = self.load_raw(train_path, valid_path, test_path)
        df = self.clean_statement_text(df)
        df = self.normalize_labels(df)
        df = self.filter_invalid_rows(df)
        df = self.parse_and_sort_by_date(df)
        df = self.build_text_modes(df)
        df = self.drop_leakage_columns(df)
        df = self.add_time_rank(df)

        print(f"\nrun_pipeline complete: {len(df)} final rows.")
        print("Official split counts (post-cleaning):")
        print(df["official_split"].value_counts().reindex(["train", "valid", "test"]))
        return df


if __name__ == "__main__":
    print("liar2_preprocessor.py loaded. Class: Liar2Preprocessor")

In [ ]:
"""
temporal_splitter.py

Class 1 of 3: splitting logic only, from da-minilm-main-model-v1-0.ipynb's
cell 4 ("Split functions: random, chronological, and rolling windows").

This is a standalone, dataset-agnostic version of the same splitting
functions Liar2Preprocessor already exposes (chronological_split,
random_stratified_split, make_rolling_windows) -- pulled out on its own
here because the model class and the experiment-runner class in this
same three-class set shouldn't need to know about Liar2Preprocessor
specifically. Any DataFrame with a `label_id` integer column works.

If you're already using Liar2Preprocessor for LIAR2 specifically, its
built-in chronological_split()/random_stratified_split()/
make_rolling_windows() are equivalent to this class's methods (same
logic, verified against the same notebook) -- use whichever is more
convenient for your call site. DAMiniLMExperimentRunner (class 3) accepts
either.

Usage
-----
    from temporal_splitter import TemporalSplitter

    splitter = TemporalSplitter()
    train_idx, valid_idx, test_idx = splitter.chronological_split(df)
    train_idx, valid_idx, test_idx = splitter.random_stratified_split(df, seed=42)
    windows = splitter.make_rolling_windows(df)
"""

from typing import Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


class TemporalSplitter:

    def __init__(self, train_ratio: float = 0.70, valid_ratio: float = 0.15, test_ratio: float = 0.15,
                 seed: int = 42):
        assert abs((train_ratio + valid_ratio + test_ratio) - 1.0) < 1e-9, \
            "train_ratio + valid_ratio + test_ratio must sum to 1.0"
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio
        self.seed = seed

    def chronological_split(self, df: pd.DataFrame):
        """Contiguous 70/15/15 split in row order. `df` must already be
        chronologically sorted (e.g. by Liar2Preprocessor's
        parse_and_sort_by_date(), or any equivalent date-ordering step)."""
        n = len(df)
        train_end = int(n * self.train_ratio)
        valid_end = int(n * (self.train_ratio + self.valid_ratio))
        return np.arange(0, train_end), np.arange(train_end, valid_end), np.arange(valid_end, n)

    def random_stratified_split(self, df: pd.DataFrame, seed: Optional[int] = None):
        """70/15/15 stratified-by-label split (label taken from `label_id`)."""
        seed = self.seed if seed is None else seed
        all_idx = np.arange(len(df))
        y = df["label_id"].values

        train_idx, temp_idx = train_test_split(
            all_idx, train_size=self.train_ratio, random_state=seed, stratify=y,
        )
        y_temp = y[temp_idx]
        valid_relative_ratio = self.valid_ratio / (self.valid_ratio + self.test_ratio)
        valid_idx, test_idx = train_test_split(
            temp_idx, train_size=valid_relative_ratio, random_state=seed, stratify=y_temp,
        )
        return np.sort(train_idx), np.sort(valid_idx), np.sort(test_idx)

    def get_split(self, df: pd.DataFrame, split_mode: str, seed: Optional[int] = None):
        if split_mode == "chronological":
            return self.chronological_split(df)
        if split_mode == "random_stratified":
            return self.random_stratified_split(df, seed=seed)
        raise ValueError(f"Unknown split_mode: {split_mode}")

    def make_rolling_windows(
        self, df: pd.DataFrame, min_train_frac: float = 0.50, valid_frac: float = 0.10,
        test_frac: float = 0.10, step_frac: float = 0.10,
    ) -> list:
        """Expanding-window temporal evaluation. `df` must already be
        chronologically sorted. Example window: train = earliest 50%,
        valid = next 10%, test = next 10%; the training window then
        expands by step_frac each iteration."""
        n = len(df)
        windows = []
        train_end = int(n * min_train_frac)
        valid_size = int(n * valid_frac)
        test_size = int(n * test_frac)
        step = max(1, int(n * step_frac))

        window_id = 0
        while train_end + valid_size + test_size <= n:
            valid_end = train_end + valid_size
            test_end = valid_end + test_size

            train_idx = np.arange(0, train_end)
            valid_idx = np.arange(train_end, valid_end)
            test_idx = np.arange(valid_end, test_end)

            windows.append({
                "window_id": window_id, "train_idx": train_idx, "valid_idx": valid_idx, "test_idx": test_idx,
                "train_end_frac": train_end / n, "valid_end_frac": valid_end / n, "test_end_frac": test_end / n,
                "n_train": len(train_idx), "n_valid": len(valid_idx), "n_test": len(test_idx),
            })
            train_end += step
            window_id += 1

        return windows


if __name__ == "__main__":
    print("temporal_splitter.py loaded. Class: TemporalSplitter")

In [ ]:
"""
da_minilm_rwc_model.py

Class 2 of 3: the model(s) only -- frozen MiniLM sentence embeddings, the
DA-MiniLM-r MLP classifier, and the frozen-embedding Logistic Regression
comparator (MiniLM-LogReg-RW). No splitting logic (see temporal_splitter
.py) and no experiment orchestration -- sample weighting, calibration,
multi-seed batteries, and bootstrap comparison (see
da_minilm_experiment_runner.py).

This class trains and predicts. It does not decide what data to train on,
how to weight it, or how to calibrate the output -- those are the
experiment runner's job, so this class stays reusable for other weighting
or calibration schemes without modification.
"""

import random
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


NUM_CLASSES_DEFAULT = 6


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class EmbeddingDataset(Dataset):
    def __init__(self, X, y=None, sample_weight=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
        if sample_weight is None:
            sample_weight = np.ones(len(X), dtype=np.float32)
        self.sample_weight = torch.tensor(sample_weight, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        item = {"x": self.X[idx], "sample_weight": self.sample_weight[idx]}
        if self.y is not None:
            item["y"] = self.y[idx]
        return item


class DAMiniLMClassifier(nn.Module):
    """The MLP head trained on frozen MiniLM embeddings."""

    def __init__(self, input_dim: int, num_classes: int = NUM_CLASSES_DEFAULT, hidden_dim: int = 256, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def predict_da_minilm_proba(model, loader, device) -> np.ndarray:
    model.eval()
    probas = []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            logits = model(x)
            probas.append(F.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(probas, axis=0)


class DAMiniLMModel:

    def __init__(
        self,
        output_dir="./da_minilm_results",
        minilm_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        device: Optional[str] = None,
        num_classes: int = NUM_CLASSES_DEFAULT,
        hidden_dim: int = 256,
        dropout: float = 0.25,
        mlp_epochs: int = 35,
        mlp_batch_size: int = 128,
        mlp_lr: float = 1e-3,
        mlp_weight_decay: float = 1e-4,
        mlp_patience: int = 6,
    ):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.minilm_model_name = minilm_model_name
        self.device = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.dropout = dropout
        self.mlp_epochs = mlp_epochs
        self.mlp_batch_size = mlp_batch_size
        self.mlp_lr = mlp_lr
        self.mlp_weight_decay = mlp_weight_decay
        self.mlp_patience = mlp_patience

        self._embedding_cache = {}

    # ---------------- Embedding extraction ----------------

    def encode_texts_minilm(self, texts: Sequence[str], cache_key: str, batch_size: int = 128) -> np.ndarray:
        """Encode texts once, cached in memory and on disk. Requires
        downloading `self.minilm_model_name` from the Hugging Face Hub the
        first time it's called for a given environment."""
        cache_path = self.output_dir / f"embeddings_{cache_key}.npy"
        if cache_key in self._embedding_cache:
            return self._embedding_cache[cache_key]
        if cache_path.exists():
            X = np.load(cache_path)
            self._embedding_cache[cache_key] = X
            print(f"Loaded cached embeddings: {cache_path} {X.shape}")
            return X

        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(self.minilm_model_name, device=str(self.device))
        X = model.encode(
            list(texts), batch_size=batch_size, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)

        np.save(cache_path, X)
        self._embedding_cache[cache_key] = X
        print(f"Saved embeddings: {cache_path} {X.shape}")
        return X

    # ---------------- DA-MiniLM MLP ----------------

    def train_da_minilm_mlp(self, X_train, y_train, w_train, X_valid, y_valid, X_test, seed: int = 42):
        """Trains the MLP head on frozen embeddings with per-sample
        weights (`w_train`), early-stopping on validation macro-F1.
        Returns (valid_proba, test_proba, best_valid_macro_f1) -- raw,
        uncalibrated probabilities; calibration is the experiment
        runner's job, not this class's."""
        set_all_seeds(seed)

        train_ds = EmbeddingDataset(X_train, y_train, w_train)
        valid_ds = EmbeddingDataset(X_valid, y_valid)
        test_ds = EmbeddingDataset(X_test, None)

        generator = torch.Generator()
        generator.manual_seed(seed)

        train_loader = DataLoader(train_ds, batch_size=self.mlp_batch_size, shuffle=True, generator=generator)
        valid_loader = DataLoader(valid_ds, batch_size=self.mlp_batch_size, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=self.mlp_batch_size, shuffle=False)

        model = DAMiniLMClassifier(
            input_dim=X_train.shape[1], num_classes=self.num_classes,
            hidden_dim=self.hidden_dim, dropout=self.dropout,
        ).to(self.device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.mlp_lr, weight_decay=self.mlp_weight_decay)

        best_state = None
        best_valid_macro_f1 = -1.0
        bad_epochs = 0

        for epoch in range(self.mlp_epochs):
            model.train()
            for batch in train_loader:
                x = batch["x"].to(self.device)
                y = batch["y"].to(self.device)
                sw = batch["sample_weight"].to(self.device)

                logits = model(x)
                per_sample_loss = F.cross_entropy(logits, y, reduction="none")
                loss = (per_sample_loss * sw).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            valid_proba = predict_da_minilm_proba(model, valid_loader, self.device)
            valid_pred = valid_proba.argmax(axis=1)
            valid_macro_f1 = f1_score(y_valid, valid_pred, average="macro", zero_division=0)

            if valid_macro_f1 > best_valid_macro_f1:
                best_valid_macro_f1 = valid_macro_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
            if bad_epochs >= self.mlp_patience:
                break

        if best_state is not None:
            model.load_state_dict(best_state)

        valid_proba = predict_da_minilm_proba(model, valid_loader, self.device)
        test_proba = predict_da_minilm_proba(model, test_loader, self.device)
        return valid_proba, test_proba, best_valid_macro_f1

    # ---------------- Frozen-embedding Logistic Regression comparator ----------------

    def train_logreg_baseline(self, X_train, y_train, w_train, X_valid, y_valid, X_test, seed: int = 42):
        """MiniLM-LogReg-RW: frozen MiniLM embeddings + Logistic
        Regression. class_weight=None because sample_weight (w_train)
        already encodes class balance if the caller included it (see
        da_minilm_experiment_runner.py's combined_sample_weights)."""
        clf = make_pipeline(
            StandardScaler(with_mean=True),
            LogisticRegression(max_iter=2000, class_weight=None, random_state=seed),
        )
        clf.fit(X_train, y_train, logisticregression__sample_weight=w_train)
        return clf.predict_proba(X_valid), clf.predict_proba(X_test)


if __name__ == "__main__":
    print("da_minilm_model.py loaded. Class: DAMiniLMModel")

In [ ]:
"""
da_minilm_experiment_runner.py

Class 3 of 3: experiment orchestration. Uses TemporalSplitter (class 1)
for splitting and DAMiniLMModel (class 2) for training/prediction -- this
class owns everything in between and around those two: metrics, sample
weighting (class-balance + recency), calibration (bias + temperature
selection), the recency-lambda grid search, multi-seed batteries,
result summarization, paired bootstrap comparison, and rolling-window
temporal evaluation.

--------------------------------------------------------------------------
A known, deliberately-preserved bug: the calibration temperature grid
--------------------------------------------------------------------------
select_best_calibration() grid-searches (tau, temperature) and keeps
whichever combination gets the best VALIDATION macro-F1. But temperature
scaling divides every class's logit by the same positive scalar -- a
monotonic transform that CANNOT change which class has the highest logit,
i.e. it cannot change argmax, i.e. it cannot change macro-F1 at all. For
any fixed tau, every temperature in temperature_grid produces the exact
same macro-F1. Because the selection loop uses strict `>` comparison,
the FIRST temperature value in the grid for the winning tau is silently
kept -- not because it's better, but because later ties never satisfy
`score > best["score"]`. This matches this project's own documented
finding ("temperature non-identifiable under macro-F1 objective; locks to
first grid value via iteration-order artifact") -- verified directly
(temperature confirmed to have zero effect on macro-F1 across the full
grid, and the first grid value confirmed to silently win). Reproduced
here byte-for-byte, commented rather than silently fixed, per this
project's practice of disclosing known bugs rather than patching them out
after the fact. Temperature DOES still matter for ECE/Brier/NLL (those
depend on the actual probabilities, not just argmax) -- it only fails to
matter for the objective this selection loop actually uses by default.
Pass objective="nll" to select_best_calibration() for an objective
temperature actually affects.
--------------------------------------------------------------------------

Usage
-----
    from temporal_splitter import TemporalSplitter
    from da_minilm_model import DAMiniLMModel
    from da_minilm_experiment_runner import DAMiniLMExperimentRunner

    splitter = TemporalSplitter()
    model = DAMiniLMModel(output_dir="./da_minilm_results")
    runner = DAMiniLMExperimentRunner(splitter=splitter, model=model)

    embeddings = {mode: model.encode_texts_minilm(df[f"model_text__{mode}"].tolist(), cache_key=mode)
                  for mode in ["claim_only", "claim_plus_safe_metadata"]}

    main_results_df, lambda_selection_df, prediction_store = runner.run_full_battery(
        df, embeddings, seeds=[42, 43, 44, 45, 46],
        text_modes=["claim_only", "claim_plus_safe_metadata"],
        split_modes=["chronological", "random_stratified"],
    )

    summary_df = runner.summarize_results(main_results_df, group_cols=["model", "text_mode", "split_mode"])
    bootstrap_df = runner.run_paired_bootstrap(prediction_store,
                                                text_modes=["claim_only", "claim_plus_safe_metadata"],
                                                split_modes=["chronological", "random_stratified"],
                                                seeds=[42, 43, 44, 45, 46])
"""

from typing import Optional, Sequence

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, log_loss,
)


NUM_CLASSES_DEFAULT = 6


# ============================================================
# Metrics
# ============================================================

def ordinal_mae(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def within_one_accuracy(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) <= 1))


def expected_calibration_error(y_true, proba, n_bins: int = 10) -> float:
    """Multiclass ECE based on confidence of the predicted class."""
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    ece = 0.0
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        bin_acc = correct[mask].mean()
        bin_conf = conf[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)
    return float(ece)


def multiclass_brier_score(y_true, proba, num_classes: int = NUM_CLASSES_DEFAULT) -> float:
    y_true = np.asarray(y_true)
    y_onehot = np.eye(num_classes)[y_true]
    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))


def safe_nll(y_true, proba, num_classes: int = NUM_CLASSES_DEFAULT) -> float:
    return float(log_loss(y_true, np.clip(proba, 1e-12, 1.0), labels=list(range(num_classes))))


def evaluate_predictions(
    y_true, y_pred, proba=None, model_name="model", split_mode=None, text_mode=None,
    seed=None, recency_lambda=None, tau=None, temperature=None, extra=None,
    num_classes: int = NUM_CLASSES_DEFAULT,
) -> dict:
    """`num_classes` MUST match proba.shape[1] -- it defaults to 6 (LIAR2)
    for backward compatibility, but this default silently breaks on any
    other label space (e.g. MultiFC's 3 classes) via a shape mismatch in
    multiclass_brier_score/safe_nll. Always pass num_classes explicitly
    when working outside the 6-class LIAR2/LIAR label space -- confirmed
    this default was a real bug when first reusing this code for MultiFC."""
    row = {
        "model": model_name, "split_mode": split_mode, "text_mode": text_mode, "seed": seed,
        "recency_lambda": recency_lambda, "tau": tau, "temperature": temperature,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "ordinal_mae": ordinal_mae(y_true, y_pred),
        "within_one_accuracy": within_one_accuracy(y_true, y_pred),
    }
    if proba is not None:
        row.update({
            "ece": expected_calibration_error(y_true, proba),
            "brier": multiclass_brier_score(y_true, proba, num_classes=num_classes),
            "nll": safe_nll(y_true, proba, num_classes=num_classes),
        })
    if extra:
        row.update(extra)
    return row


# ============================================================
# Sample weighting
# ============================================================

def normalize_weights(w, min_weight: float = 0.05, max_weight: float = 20.0) -> np.ndarray:
    w = np.asarray(w, dtype=float)
    w = np.clip(w, min_weight, max_weight)
    return w / (np.mean(w) + 1e-12)


def effective_number_class_weights(y, beta: float = 0.999, num_classes: int = NUM_CLASSES_DEFAULT) -> np.ndarray:
    """Per-class weights based on the effective number of samples (Cui et al., 2019)."""
    counts = np.bincount(np.asarray(y, dtype=int), minlength=num_classes).astype(float)
    weights = np.ones(num_classes, dtype=float)
    for c in range(num_classes):
        n = counts[c]
        if n > 0:
            effective_num = (1.0 - beta ** n) / (1.0 - beta)
            weights[c] = 1.0 / effective_num
    return weights / (weights.mean() + 1e-12)


def class_sample_weights(y_train, beta: float = 0.999, num_classes: int = NUM_CLASSES_DEFAULT) -> np.ndarray:
    class_w = effective_number_class_weights(y_train, beta=beta, num_classes=num_classes)
    return normalize_weights(class_w[np.asarray(y_train, dtype=int)])


def recency_sample_weights(time_rank_train, lam: float = 0.0) -> np.ndarray:
    """Exponential recency weighting: age = max_train_time - sample_time;
    higher lambda gives more weight to recent training samples."""
    t = np.asarray(time_rank_train, dtype=float)
    age = t.max() - t
    w = np.exp(-float(lam) * age * 10.0)
    return normalize_weights(w)


def combined_sample_weights(y_train, time_rank_train, recency_lambda: float = 0.0,
                             beta: float = 0.999, num_classes: int = NUM_CLASSES_DEFAULT) -> np.ndarray:
    w_class = class_sample_weights(y_train, beta=beta, num_classes=num_classes)
    w_recency = recency_sample_weights(time_rank_train, recency_lambda)
    return normalize_weights(w_class * w_recency)


# ============================================================
# Calibration
# ============================================================

def softmax_np(logits) -> np.ndarray:
    logits = np.asarray(logits, dtype=float)
    logits = logits - logits.max(axis=1, keepdims=True)
    expv = np.exp(logits)
    return expv / (expv.sum(axis=1, keepdims=True) + 1e-12)


def proba_to_logits(proba, eps: float = 1e-12) -> np.ndarray:
    return np.log(np.clip(proba, eps, 1.0))


def logit_adjustment_bias(y_train, num_classes: int = NUM_CLASSES_DEFAULT, tau: float = 1.0, eps: float = 1e-12) -> np.ndarray:
    counts = np.bincount(y_train, minlength=num_classes).astype(float)
    prior = counts / counts.sum()
    return tau * np.log(prior + eps)


def apply_bias_and_temperature(proba, bias=None, temperature: float = 1.0) -> np.ndarray:
    logits = proba_to_logits(proba)
    if bias is not None:
        logits = logits - bias.reshape(1, -1)
    logits = logits / max(float(temperature), 1e-6)
    return softmax_np(logits)


def select_best_calibration(
    valid_proba, y_valid, y_train, objective: str = "macro_f1",
    tau_grid: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
    temperature_grid: Sequence[float] = (0.75, 1.0, 1.25, 1.5, 2.0),
    num_classes: int = NUM_CLASSES_DEFAULT,
) -> dict:
    """Select tau and temperature using validation only.

    NOTE: with objective="macro_f1" (the default), temperature has no
    effect on the selected argmax and is therefore effectively
    unidentifiable -- see module docstring. Use objective="nll" for an
    objective temperature actually affects, or use
    select_best_calibration_two_stage() below instead of this function.
    """
    best = {"tau": 0.0, "temperature": 1.0, "score": -1e9}
    for tau in tau_grid:
        bias = logit_adjustment_bias(y_train, tau=tau, num_classes=num_classes)
        for temp in temperature_grid:
            calibrated = apply_bias_and_temperature(valid_proba, bias=bias, temperature=temp)
            pred = calibrated.argmax(axis=1)
            if objective == "macro_f1":
                score = f1_score(y_valid, pred, average="macro", zero_division=0)
            elif objective == "nll":
                score = -safe_nll(y_valid, calibrated, num_classes=num_classes)
            else:
                raise ValueError(f"Unknown objective: {objective}")
            if score > best["score"]:
                best = {"tau": tau, "temperature": temp, "score": float(score)}
    return best


def select_best_calibration_two_stage(
    valid_proba, y_valid, y_train,
    tau_grid: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
    temperature_grid: Sequence[float] = (0.75, 1.0, 1.25, 1.5, 2.0),
    num_classes: int = NUM_CLASSES_DEFAULT,
) -> dict:
    """The calibration-bug fix. select_best_calibration() jointly grid-
    searches (tau, temperature) under a single objective -- but temperature
    has zero effect on argmax, so under objective="macro_f1" it can never
    be meaningfully selected (see module docstring; confirmed by direct
    test). This function decouples the two selections onto the objective
    each one actually affects:

    Stage 1 -- select tau under macro_f1 (tau/logit-adjustment DOES shift
               the decision boundary via the class-prior bias, so this
               objective is meaningful here).
    Stage 2 -- with tau fixed at its stage-1 value, select temperature
               under NLL (temperature doesn't move argmax but does reshape
               the probability distribution, so NLL -- which depends on
               the full distribution, not just the top class -- is
               sensitive to it. This is the "objective each dimension can
               actually see" pairing that was missing from the joint
               search.)

    Returns {"tau", "temperature", "macro_f1_at_best_tau", "nll_at_best_temperature"}.
    """
    # Stage 1: tau, evaluated at temperature=1.0 (irrelevant to macro_f1 anyway).
    best_tau, best_tau_score = 0.0, -1e9
    for tau in tau_grid:
        bias = logit_adjustment_bias(y_train, tau=tau, num_classes=num_classes)
        calibrated = apply_bias_and_temperature(valid_proba, bias=bias, temperature=1.0)
        score = f1_score(y_valid, calibrated.argmax(axis=1), average="macro", zero_division=0)
        if score > best_tau_score:
            best_tau, best_tau_score = tau, float(score)

    # Stage 2: temperature, with tau fixed, evaluated under NLL.
    bias = logit_adjustment_bias(y_train, tau=best_tau, num_classes=num_classes)
    best_temp, best_temp_score = 1.0, 1e9
    for temp in temperature_grid:
        calibrated = apply_bias_and_temperature(valid_proba, bias=bias, temperature=temp)
        nll = safe_nll(y_valid, calibrated, num_classes=num_classes)
        if nll < best_temp_score:
            best_temp, best_temp_score = temp, float(nll)

    return {
        "tau": best_tau, "temperature": best_temp,
        "macro_f1_at_best_tau": best_tau_score, "nll_at_best_temperature": best_temp_score,
    }


# ============================================================
# Statistical comparison
# ============================================================

def paired_bootstrap_macro_f1_diff(y_true, y_pred_a, y_pred_b, n_boot: int = 2000, seed: int = 42) -> dict:
    rng = np.random.RandomState(seed)
    y_true, y_pred_a, y_pred_b = np.asarray(y_true), np.asarray(y_pred_a), np.asarray(y_pred_b)
    n = len(y_true)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        f1_a = f1_score(y_true[idx], y_pred_a[idx], average="macro", zero_division=0)
        f1_b = f1_score(y_true[idx], y_pred_b[idx], average="macro", zero_division=0)
        diffs[b] = f1_a - f1_b

    ci_lower, ci_upper = np.percentile(diffs, [2.5, 97.5])
    prop_le_0 = float(np.mean(diffs <= 0))
    prop_ge_0 = float(np.mean(diffs >= 0))
    p_value = min(2.0 * min(prop_le_0, prop_ge_0), 1.0)

    return {
        "mean_diff": float(diffs.mean()), "ci_lower_95": float(ci_lower), "ci_upper_95": float(ci_upper),
        "p_value": p_value, "significant_at_95": bool(ci_lower > 0 or ci_upper < 0),
    }


# ============================================================
# Experiment runner
# ============================================================

class DAMiniLMExperimentRunner:

    def __init__(
        self,
        splitter,
        model,
        recency_lambdas: Sequence[float] = (0.0, 0.05, 0.10, 0.20, 0.30),
        class_weight_beta: float = 0.999,
        temperature_grid: Sequence[float] = (0.75, 1.0, 1.25, 1.5, 2.0),
        tau_grid: Sequence[float] = (0.0, 0.5, 1.0, 1.5, 2.0),
        num_classes: int = NUM_CLASSES_DEFAULT,
        output_dir=None,
    ):
        """`splitter` should be a TemporalSplitter instance (class 1) --
        used for chronological_split / random_stratified_split /
        make_rolling_windows. `model` should be a DAMiniLMModel instance
        (class 2) -- used for train_da_minilm_mlp / train_logreg_baseline.
        Neither is reimplemented here."""
        self.splitter = splitter
        self.model = model
        self.recency_lambdas = recency_lambdas
        self.class_weight_beta = class_weight_beta
        self.temperature_grid = temperature_grid
        self.tau_grid = tau_grid
        self.num_classes = num_classes
        self.output_dir = output_dir or getattr(model, "output_dir", None)

    # ---------------- Single-run orchestration ----------------

    def run_single_experiment(
        self, X, y, time_rank, train_idx, valid_idx, test_idx,
        text_mode: str, split_mode: str, seed: int, model_kind: str = "da_minilm_mlp",
        use_reweighting: bool = True, use_calibration: bool = True,
        calibration_mode: str = "joint_macro_f1", model_name_override: Optional[str] = None,
    ):
        """model_kind: "da_minilm_mlp" (MLP head) or "minilm_logreg"
        (Logistic Regression head) -- the architecture dimension of the
        component ablation.

        use_reweighting: if False, sample weights are uniform (no class
        balancing, no recency weighting) and self.recency_lambdas is
        skipped entirely -- this is the ablation's "Uniform" condition
        for whichever architecture is selected (e.g. model_kind=
        "minilm_logreg" + use_reweighting=False == Uniform-LR).

        use_calibration: if False, raw (uncalibrated) argmax predictions
        are used directly -- no tau/temperature search at all. This
        isolates the calibration step's contribution.

        calibration_mode: only relevant when use_calibration=True.
            "joint_macro_f1" -- the ORIGINAL notebook's approach: joint
                (tau, temperature) grid search under macro_f1. Preserved
                for exact reproduction; see module docstring re: the
                temperature non-identifiability this causes.
            "two_stage" -- the BUG FIX: tau selected under macro_f1,
                temperature separately selected under NLL. Use this for
                new results; use "joint_macro_f1" only to reproduce prior
                numbers exactly.

        Returns (row, lambda_df, pred_test, test_cal). `lambda_df` has a
        single row (recency_lambda=0.0) when use_reweighting=False, since
        there is no lambda grid to search in that case.
        """
        X_train, X_valid, X_test = X[train_idx], X[valid_idx], X[test_idx]
        y_train, y_valid, y_test = y[train_idx], y[valid_idx], y[test_idx]
        t_train = time_rank[train_idx]

        recency_lambdas = self.recency_lambdas if use_reweighting else (0.0,)

        lambda_candidates = []
        stored = {}

        for rec_lam in recency_lambdas:
            if use_reweighting:
                w_train = combined_sample_weights(
                    y_train, t_train, recency_lambda=rec_lam,
                    beta=self.class_weight_beta, num_classes=self.num_classes,
                )
            else:
                w_train = np.ones(len(y_train), dtype=float)

            if model_kind == "da_minilm_mlp":
                valid_proba, test_proba, raw_valid_f1 = self.model.train_da_minilm_mlp(
                    X_train, y_train, w_train, X_valid, y_valid, X_test, seed=seed,
                )
                default_model_name = "DA-MiniLM-RWAC" if use_reweighting else "Uniform-MLP"
            elif model_kind == "minilm_logreg":
                valid_proba, test_proba = self.model.train_logreg_baseline(
                    X_train, y_train, w_train, X_valid, y_valid, X_test, seed=seed,
                )
                raw_valid_f1 = f1_score(y_valid, valid_proba.argmax(axis=1), average="macro", zero_division=0)
                default_model_name = "MiniLM-LogReg-RW" if use_reweighting else "Uniform-LR"
            else:
                raise ValueError(f"Unknown model_kind: {model_kind}")
            model_name = model_name_override or default_model_name

            if use_calibration:
                if calibration_mode == "joint_macro_f1":
                    calib = select_best_calibration(
                        valid_proba, y_valid, y_train, objective="macro_f1",
                        tau_grid=self.tau_grid, temperature_grid=self.temperature_grid, num_classes=self.num_classes,
                    )
                elif calibration_mode == "two_stage":
                    calib_raw = select_best_calibration_two_stage(
                        valid_proba, y_valid, y_train,
                        tau_grid=self.tau_grid, temperature_grid=self.temperature_grid, num_classes=self.num_classes,
                    )
                    calib = {"tau": calib_raw["tau"], "temperature": calib_raw["temperature"]}
                else:
                    raise ValueError(f"Unknown calibration_mode: {calibration_mode}")
            else:
                calib = {"tau": 0.0, "temperature": 1.0}

            valid_cal = apply_bias_and_temperature(
                valid_proba,
                bias=logit_adjustment_bias(y_train, tau=calib["tau"], num_classes=self.num_classes),
                temperature=calib["temperature"],
            )
            valid_macro_f1 = f1_score(y_valid, valid_cal.argmax(axis=1), average="macro", zero_division=0)

            lambda_candidates.append({
                "recency_lambda": rec_lam, "valid_macro_f1": valid_macro_f1,
                "raw_valid_macro_f1": raw_valid_f1, "tau": calib["tau"], "temperature": calib["temperature"],
            })
            stored[rec_lam] = (valid_proba, test_proba, calib)

        lambda_df = pd.DataFrame(lambda_candidates).sort_values("valid_macro_f1", ascending=False)
        best_lambda = float(lambda_df.iloc[0]["recency_lambda"])
        valid_proba, test_proba, calib = stored[best_lambda]

        bias = logit_adjustment_bias(y_train, tau=calib["tau"], num_classes=self.num_classes)
        test_cal = apply_bias_and_temperature(test_proba, bias=bias, temperature=calib["temperature"])
        pred_test = test_cal.argmax(axis=1)

        row = evaluate_predictions(
            y_test, pred_test, proba=test_cal, model_name=model_name, split_mode=split_mode,
            text_mode=text_mode, seed=seed, recency_lambda=best_lambda, tau=calib["tau"],
            temperature=calib["temperature"], num_classes=self.num_classes,
            extra={
                "n_train": len(train_idx), "n_valid": len(valid_idx), "n_test": len(test_idx),
                "selected_valid_macro_f1": float(lambda_df.iloc[0]["valid_macro_f1"]),
                "use_reweighting": use_reweighting, "use_calibration": use_calibration,
                "calibration_mode": calibration_mode if use_calibration else None,
            },
        )
        return row, lambda_df, pred_test, test_cal

    # ---------------- Component ablation ----------------

    def run_component_ablation(
        self, X, y, time_rank, train_idx, valid_idx, test_idx,
        text_mode: str, split_mode: str, seed: int, calibration_mode: str = "two_stage",
    ) -> pd.DataFrame:
        """The four ablation cells isolating each design choice's
        contribution, plus the two full models for reference:

            Uniform-LR         : LogReg, no reweighting, no calibration
            LogReg + RW         : LogReg, reweighting, no calibration
            MiniLM-LogReg-RW    : LogReg, reweighting, calibration (full)
            Uniform-MLP         : MLP,    no reweighting, no calibration
            MLP + RW            : MLP,    reweighting, no calibration
            DA-MiniLM-RWAC      : MLP,    reweighting, calibration (full)

        Reading down each architecture's three rows isolates reweighting's
        contribution (row 2 minus row 1) and calibration's contribution
        (row 3 minus row 2), holding architecture fixed. Reading across
        LogReg vs MLP at matched reweighting/calibration settings isolates
        the architecture's own contribution.
        """
        cells = [
            ("minilm_logreg", False, False, "Uniform-LR"),
            ("minilm_logreg", True, False, "LogReg + RW (no calibration)"),
            ("minilm_logreg", True, True, "MiniLM-LogReg-RW"),
            ("da_minilm_mlp", False, False, "Uniform-MLP"),
            ("da_minilm_mlp", True, False, "MLP + RW (no calibration)"),
            ("da_minilm_mlp", True, True, "DA-MiniLM-RWAC"),
        ]
        rows = []
        for model_kind, use_rw, use_calib, label in cells:
            print(f"running ablation cell: {label}")
            row, _, _, _ = self.run_single_experiment(
                X, y, time_rank, train_idx, valid_idx, test_idx,
                text_mode=text_mode, split_mode=split_mode, seed=seed, model_kind=model_kind,
                use_reweighting=use_rw, use_calibration=use_calib,
                calibration_mode=calibration_mode, model_name_override=label,
            )
            rows.append(row)
            print(f"  macro_f1={row['macro_f1']:.4f}")
        return pd.DataFrame(rows)

    # ---------------- Full battery ----------------

    def run_full_battery(
        self, df: pd.DataFrame, embeddings: dict, seeds: Sequence[int],
        text_modes: Sequence[str], split_modes: Sequence[str],
        model_kinds: Sequence[str] = ("minilm_logreg", "da_minilm_mlp"),
        calibration_mode: str = "two_stage",
    ):
        """`embeddings` must be a dict {text_mode: np.ndarray} (from
        model.encode_texts_minilm()). Splitting is delegated to
        self.splitter -- NOT reimplemented here. `df` must have a reset
        integer index (positions == train_idx/valid_idx/test_idx from the
        split functions), a `label_id` column, and a `time_rank` column.

        BUG FIX: this method previously did not pass calibration_mode to
        run_single_experiment() at all, so every call silently used
        run_single_experiment()'s own default of "joint_macro_f1" -- the
        calibration scheme where temperature has no effect on macro-F1 and
        is therefore never meaningfully selected (see
        select_best_calibration()'s docstring). run_component_ablation()
        already defaulted to the fixed "two_stage" scheme; run_full_battery
        (the method that actually produces your main/headline results) did
        not. Now it does, by default."""
        y_all = df["label_id"].values.astype(int)
        time_rank_all = df["time_rank"].values.astype(float)

        all_rows, all_lambda_rows = [], []
        prediction_store = {}

        for text_mode in text_modes:
            X = embeddings[text_mode]
            for split_mode in split_modes:
                for seed in seeds:
                    train_idx, valid_idx, test_idx = self.splitter.get_split(df, split_mode, seed=seed)

                    for model_kind in model_kinds:
                        print(f"running [{model_kind}] text={text_mode} split={split_mode} seed={seed}")
                        row, lambda_df, pred, proba = self.run_single_experiment(
                            X, y_all, time_rank_all, train_idx, valid_idx, test_idx,
                            text_mode=text_mode, split_mode=split_mode, seed=seed, model_kind=model_kind,
                            calibration_mode=calibration_mode,
                        )
                        all_rows.append(row)
                        for _, r in lambda_df.iterrows():
                            all_lambda_rows.append({
                                "model": row["model"], "text_mode": text_mode, "split_mode": split_mode,
                                "seed": seed, **r.to_dict(),
                            })
                        prediction_store[(row["model"], text_mode, split_mode, seed)] = {
                            "y_test": y_all[test_idx], "pred": pred, "proba": proba,
                        }
                        print(f"  macro_f1={row['macro_f1']:.4f}, ece={row['ece']:.4f}, lambda={row['recency_lambda']}")

        main_results_df = pd.DataFrame(all_rows)
        lambda_selection_df = pd.DataFrame(all_lambda_rows)

        if self.output_dir is not None:
            main_results_df.to_csv(f"{self.output_dir}/main_da_minilm_results.csv", index=False)
            lambda_selection_df.to_csv(f"{self.output_dir}/lambda_selection_validation_results.csv", index=False)

        return main_results_df, lambda_selection_df, prediction_store

    # ---------------- Summaries and comparisons ----------------

    def summarize_results(self, results_df: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
        metrics = ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy", "ece", "brier", "nll"]
        agg = {}
        for m in metrics:
            if m in results_df.columns:
                agg[f"{m}_mean"] = (m, "mean")
                agg[f"{m}_std"] = (m, "std")
                agg[f"{m}_min"] = (m, "min")
                agg[f"{m}_max"] = (m, "max")
        return results_df.groupby(group_cols).agg(**agg, n_runs=("macro_f1", "count")).reset_index()

    def run_paired_bootstrap(
        self, prediction_store: dict, text_modes: Sequence[str], split_modes: Sequence[str],
        seeds: Sequence[int], model_a: str = "DA-MiniLM-RWAC", model_b: str = "MiniLM-LogReg-RW",
        n_boot: int = 2000,
    ) -> pd.DataFrame:
        rows = []
        for text_mode in text_modes:
            for split_mode in split_modes:
                for seed in seeds:
                    key_a = (model_a, text_mode, split_mode, seed)
                    key_b = (model_b, text_mode, split_mode, seed)
                    if key_a not in prediction_store or key_b not in prediction_store:
                        continue
                    y_test = prediction_store[key_a]["y_test"]
                    boot = paired_bootstrap_macro_f1_diff(
                        y_test, prediction_store[key_a]["pred"], prediction_store[key_b]["pred"],
                        n_boot=n_boot, seed=seed,
                    )
                    rows.append({
                        "comparison": f"{model_a} minus {model_b}", "text_mode": text_mode,
                        "split_mode": split_mode, "seed": seed, **boot,
                    })
        bootstrap_df = pd.DataFrame(rows)
        if self.output_dir is not None:
            bootstrap_df.to_csv(f"{self.output_dir}/paired_bootstrap_results.csv", index=False)
        return bootstrap_df

    def run_rolling_window_evaluation(
        self, df: pd.DataFrame, embeddings: dict, text_mode: str, seeds: Sequence[int],
        model_kind: str = "da_minilm_mlp", rolling_windows_kwargs: Optional[dict] = None,
    ) -> pd.DataFrame:
        """Rolling windows are built by self.splitter.make_rolling_windows()
        -- not reimplemented here."""
        rolling_windows_kwargs = rolling_windows_kwargs or {}
        windows = self.splitter.make_rolling_windows(df, **rolling_windows_kwargs)

        y_all = df["label_id"].values.astype(int)
        time_rank_all = df["time_rank"].values.astype(float)
        X = embeddings[text_mode]

        rows = []
        for w in windows:
            for seed in seeds:
                print(f"running [rolling] window={w['window_id']} seed={seed}")
                row, lambda_df, pred, proba = self.run_single_experiment(
                    X, y_all, time_rank_all, w["train_idx"], w["valid_idx"], w["test_idx"],
                    text_mode=text_mode, split_mode="rolling_window", seed=seed, model_kind=model_kind,
                )
                row.update({
                    "window_id": w["window_id"], "train_end_frac": w["train_end_frac"],
                    "valid_end_frac": w["valid_end_frac"], "test_end_frac": w["test_end_frac"],
                })
                rows.append(row)
                print(f"  macro_f1={row['macro_f1']:.4f}, ece={row['ece']:.4f}, lambda={row['recency_lambda']}")

        rolling_results_df = pd.DataFrame(rows)
        if self.output_dir is not None:
            rolling_results_df.to_csv(f"{self.output_dir}/rolling_window_results.csv", index=False)
        return rolling_results_df


if __name__ == "__main__":
    print("da_minilm_experiment_runner.py loaded. Class: DAMiniLMExperimentRunner")

In [ ]:
# ============================================================================
# PASTE PROJECT CLASS FILES HERE (in order):
#   1) liar2_preprocessor.py           (class Liar2Preprocessor)
#   2) temporal_splitter.py            (class TemporalSplitter)
#   3) da_minilm_model.py              (class DAMiniLMModel + set_all_seeds)
#   4) da_minilm_experiment_runner.py  (class DAMiniLMExperimentRunner + helpers)
# ============================================================================

# <<< PASTE liar2_preprocessor.py BELOW >>>


# <<< PASTE temporal_splitter.py BELOW >>>


# <<< PASTE da_minilm_model.py BELOW >>>


# <<< PASTE da_minilm_experiment_runner.py BELOW >>>


In [ ]:
# Sanity check: confirm the four required classes are defined before continuing.
_required = ["Liar2Preprocessor", "TemporalSplitter", "DAMiniLMModel", "DAMiniLMExperimentRunner"]
_missing = [name for name in _required if name not in dir()]
assert not _missing, (
    f"Missing class definitions: {_missing}. "
    "Paste the corresponding project .py files into the cell above."
)
print("All four project classes are defined:", ", ".join(_required))

## Section 2 — Label-collapse definitions

The 6-class canonical order is fixed by `Liar2Preprocessor.LABEL_ORDER`:

| id | 0 | 1 | 2 | 3 | 4 | 5 |
|----|---|---|---|---|---|---|
| label | pants-fire | false | barely-true | half-true | mostly-true | true |

All collapse maps below are keyed on that integer `label_id`.

| 6-class | → 3-class | → 2-class (primary) | → 2-class (robustness) |
|---|---|---|---|
| pants-fire (0) | false | false | false |
| false (1) | false | false | false |
| barely-true (2) | mixed | false | false |
| half-true (3) | mixed | **true** | **false** |
| mostly-true (4) | true | true | true |
| true (5) | true | true | true |

In [ ]:
# 3-class: {false, mixed, true} -- identical taxonomy to collapsed MultiFC
SIXCLASS_TO_THREE = {0: 0, 1: 0, 2: 1, 3: 1, 4: 2, 5: 2}
THREECLASS_ORDER  = ["false", "mixed", "true"]

# 2-class: {false, true}, half-true on the TRUE side (LIAR convention)
SIXCLASS_TO_TWO   = {0: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1}
TWOCLASS_ORDER    = ["false", "true"]

# Robustness variant: half-true on the FALSE side
SIXCLASS_TO_TWO_HT_FALSE = {0: 0, 1: 0, 2: 0, 3: 0, 4: 1, 5: 1}

# Identity (reference 6-class run)
SIXCLASS_IDENTITY = {i: i for i in range(6)}
SIXCLASS_ORDER    = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]

COLLAPSE_SCHEMES = {
    "6class":                {"map": SIXCLASS_IDENTITY,        "order": SIXCLASS_ORDER,   "num_classes": 6},
    "3class":                {"map": SIXCLASS_TO_THREE,        "order": THREECLASS_ORDER, "num_classes": 3},
    "2class":                {"map": SIXCLASS_TO_TWO,          "order": TWOCLASS_ORDER,   "num_classes": 2},
    "2class_halftrue_false": {"map": SIXCLASS_TO_TWO_HT_FALSE, "order": TWOCLASS_ORDER,   "num_classes": 2},
}


class LabelCollapser:
    """Remaps an already-cleaned LIAR2 DataFrame's `label_id` to a coarser
    granularity, leaving statement text, date order, time_rank and the
    restricted-metadata text modes untouched. The original 6-class labels are
    retained as `label_id_6class` for auditing."""

    def __init__(self, scheme: str):
        if scheme not in COLLAPSE_SCHEMES:
            raise ValueError(f"scheme must be one of {list(COLLAPSE_SCHEMES)}, got {scheme!r}")
        cfg = COLLAPSE_SCHEMES[scheme]
        self.scheme = scheme
        self.map = cfg["map"]
        self.label_order = cfg["order"]
        self.num_classes = cfg["num_classes"]

    def transform(self, df: pd.DataFrame, label_id_col: str = "label_id") -> pd.DataFrame:
        if label_id_col not in df.columns:
            raise KeyError(f"{label_id_col!r} not in DataFrame columns")
        out = df.copy()
        six = out[label_id_col].astype(int)
        unknown = set(six.unique()) - set(self.map)
        if unknown:
            raise ValueError(f"label_id values {sorted(unknown)} absent from {self.scheme} map (expected 0..5)")
        out["label_id_6class"] = six.values
        out[label_id_col] = six.map(self.map).astype(int).values
        out["label_collapsed"] = out[label_id_col].map(
            {i: n for i, n in enumerate(self.label_order)}).values
        return out

    def describe_collapse(self, df: pd.DataFrame, label_id_col: str = "label_id") -> pd.DataFrame:
        six_col = "label_id_6class" if "label_id_6class" in df.columns else label_id_col
        rows = []
        for sid in range(6):
            n = int((df[six_col] == sid).sum())
            cid = self.map[sid]
            rows.append({"six_id": sid, "six_label": SIXCLASS_ORDER[sid],
                         "collapsed_id": cid, "collapsed_label": self.label_order[cid],
                         "n_rows": n})
        tbl = pd.DataFrame(rows)
        totals = (tbl.groupby(["collapsed_id", "collapsed_label"])["n_rows"].sum()
                     .reset_index().rename(columns={"n_rows": "collapsed_total"}))
        return tbl.merge(totals, on=["collapsed_id", "collapsed_label"])


# quick self-test on synthetic ids
_demo = pd.DataFrame({"label_id": list(range(6))})
for _s in COLLAPSE_SCHEMES:
    _c = LabelCollapser(_s).transform(_demo)
    print(f"{_s:24s} -> {_c['label_id'].tolist()}  (K={LabelCollapser(_s).num_classes})")

## Section 3 — Load and preprocess LIAR2 (once)

Runs the standard preprocessing pipeline, then applies the project's canonical
**dedup → chronological reset** so row positions match the reported cohort. The
resulting `df6` (6-class) is the single source of rows every collapse scheme
reuses.

In [ ]:
prep = Liar2Preprocessor()
df6 = prep.run_pipeline(LIAR2_TRAIN, LIAR2_VALID, LIAR2_TEST)

# Canonical dedup (keep_first) to reproduce reported cohort sizes, then
# re-establish chronological row order + time_rank on the deduped frame.
try:
    _audit = prep.audit_duplicate_statements(df6)
    df6 = prep.drop_duplicate_statements(df6, policy="keep_first", audit_df=_audit)
    df6 = prep.parse_and_sort_by_date(df6)   # re-sort after dropping rows
    df6 = prep.add_time_rank(df6)            # recompute time_rank over final rows
    print("Applied dedup(keep_first) + chronological re-sort + time_rank.")
except Exception as e:
    print("NOTE: dedup step skipped or unavailable ->", repr(e))

df6 = df6.reset_index(drop=True)   # positions must align with split indices

assert "label_id" in df6.columns and "time_rank" in df6.columns
assert f"model_text__{TEXT_MODE}" in df6.columns, (
    f"Expected column model_text__{TEXT_MODE} not found. "
    f"Available model_text columns: {[c for c in df6.columns if c.startswith('model_text__')]}"
)
print(f"\nFinal LIAR2 rows: {len(df6)}")
print("6-class label_id distribution:")
print(df6["label_id"].value_counts().sort_index()
      .rename(index={i: n for i, n in enumerate(SIXCLASS_ORDER)}))

## Section 4 — Compute frozen MiniLM embeddings (once, reused everywhere)

Embeddings depend only on the text, not on the labels, so they are computed a single
time and shared across all four granularities. This is the single most expensive step;
caching it here avoids re-encoding four times over.

In [ ]:
# A shared model instance ONLY for encoding (num_classes irrelevant for encode).
_encoder = DAMiniLMModel(output_dir=OUTPUT_DIR, num_classes=6)

_text_modes_to_embed = [TEXT_MODE] + (["claim_only"] if ALSO_RUN_CLAIM_ONLY else [])

embeddings = {}
for _mode in _text_modes_to_embed:
    _col = f"model_text__{_mode}"
    print(f"Encoding text mode: {_mode}  ({_col})")
    embeddings[_mode] = _encoder.encode_texts_minilm(
        df6[_col].tolist(), cache_key=f"liar2_collapse_{_mode}"
    )
    print(f"  -> embedding shape: {embeddings[_mode].shape}")

TEXT_MODES = list(embeddings.keys())
print("\nEmbeddings ready for text modes:", TEXT_MODES)

## Section 5 — Run the collapse battery

For each scheme we build a collapsed copy of the frame, then drive the **existing**
`DAMiniLMExperimentRunner.run_full_battery` with the correct `num_classes`. That method
already sweeps both `model_kind`s per seed:

- `da_minilm_mlp` + reweighting → **DA-MiniLM-RWAC**
- `minilm_logreg` + reweighting → **MiniLM-LogReg-RW**

We additionally run the **Uniform-LR** condition (`minilm_logreg`, reweighting off) via
`run_single_experiment`, so all three comparators appear at every granularity. Weights
are recomputed inside the runner from each scheme's collapsed priors; embeddings are
reused unchanged.

`two_stage` calibration (the bug-fixed scheme) is used throughout.

In [ ]:
def build_runner(num_classes):
    """Fresh model + runner configured for a given class count. A new model is
    required so num_classes is correct for the MLP head and for Brier/NLL; a new
    runner so class-balance weights are recomputed on the collapsed priors."""
    model = DAMiniLMModel(output_dir=f"{OUTPUT_DIR}/K{num_classes}", num_classes=num_classes)
    # reuse the already-computed embedding cache so we never re-encode
    model._embedding_cache = _encoder._embedding_cache
    splitter = TemporalSplitter(train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15)
    runner = DAMiniLMExperimentRunner(
        splitter=splitter, model=model, num_classes=num_classes,
        output_dir=f"{OUTPUT_DIR}/K{num_classes}",
    )
    return model, splitter, runner


all_main_rows = []
all_bootstrap_rows = []
prediction_stores = {}      # scheme -> prediction_store from run_full_battery

for scheme in SCHEMES:
    cfg = COLLAPSE_SCHEMES[scheme]
    K = cfg["num_classes"]
    print("\n" + "=" * 72)
    print(f"SCHEME = {scheme}   (num_classes = {K})")
    print("=" * 72)

    collapser = LabelCollapser(scheme)
    df_c = collapser.transform(df6).reset_index(drop=True)
    print(collapser.describe_collapse(df_c).to_string(index=False))

    os.makedirs(f"{OUTPUT_DIR}/K{K}", exist_ok=True)
    model, splitter, runner = build_runner(K)

    # ---- DA-MiniLM-RWAC + MiniLM-LogReg-RW via the standard battery ----
    main_df, lambda_df, pred_store = runner.run_full_battery(
        df=df_c, embeddings=embeddings, seeds=SEEDS,
        text_modes=TEXT_MODES, split_modes=[SPLIT_MODE],
        model_kinds=("minilm_logreg", "da_minilm_mlp"),
        calibration_mode="two_stage",
    )

    # ---- Uniform-LR (unweighted frozen-embedding LogReg) ----
    y_all = df_c["label_id"].values.astype(int)
    t_all = df_c["time_rank"].values.astype(float)
    for _mode in TEXT_MODES:
        X = embeddings[_mode]
        for seed in SEEDS:
            tr, va, te = splitter.get_split(df_c, SPLIT_MODE, seed=seed)
            row_u, _, pred_u, proba_u = runner.run_single_experiment(
                X, y_all, t_all, tr, va, te,
                text_mode=_mode, split_mode=SPLIT_MODE, seed=seed,
                model_kind="minilm_logreg", use_reweighting=False,
                use_calibration=True, calibration_mode="two_stage",
            )
            main_df = pd.concat([main_df, pd.DataFrame([row_u])], ignore_index=True)
            pred_store[(row_u["model"], _mode, SPLIT_MODE, seed)] = {
                "y_test": y_all[te], "pred": pred_u, "proba": proba_u,
            }

    main_df["scheme"] = scheme
    main_df["num_classes"] = K
    all_main_rows.append(main_df)
    prediction_stores[scheme] = pred_store

    # ---- Paired bootstrap: DA-MiniLM-RWAC minus MiniLM-LogReg-RW ----
    boot_df = runner.run_paired_bootstrap(
        pred_store, text_modes=TEXT_MODES, split_modes=[SPLIT_MODE],
        seeds=SEEDS, model_a="DA-MiniLM-RWAC", model_b="MiniLM-LogReg-RW",
        n_boot=N_BOOT,
    )
    if len(boot_df):
        boot_df["scheme"] = scheme
        boot_df["num_classes"] = K
        all_bootstrap_rows.append(boot_df)

main_results = pd.concat(all_main_rows, ignore_index=True)
bootstrap_results = (pd.concat(all_bootstrap_rows, ignore_index=True)
                     if all_bootstrap_rows else pd.DataFrame())

main_results.to_csv(f"{OUTPUT_DIR}/collapse_main_results.csv", index=False)
if len(bootstrap_results):
    bootstrap_results.to_csv(f"{OUTPUT_DIR}/collapse_bootstrap_results.csv", index=False)

print("\nAll schemes complete.")
print("Main results rows:", len(main_results))

## Section 6 — Summary tables

Per-scheme, per-model means ± std across seeds, plus the headline
DA-MiniLM-RWAC − MiniLM-LogReg-RW effect size and its 95% bootstrap CI at each
granularity. The lead column is macro-F1; the absolute effect size and CI are
reported before any significance label, per the project's reporting discipline.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---- Per-model summary across seeds ----
summary = (main_results
    .groupby(["scheme", "num_classes", "model", "text_mode"])
    .agg(macro_f1_mean=("macro_f1", "mean"),
         macro_f1_std =("macro_f1", "std"),
         accuracy_mean=("accuracy", "mean"),
         macro_recall_mean=("macro_recall", "mean"),
         ece_mean=("ece", "mean"),
         n_runs=("macro_f1", "count"))
    .reset_index()
    .sort_values(["num_classes", "text_mode", "macro_f1_mean"], ascending=[False, True, False]))

# order schemes 6 -> 3 -> 2 -> 2(robust)
_scheme_rank = {"6class": 0, "3class": 1, "2class": 2, "2class_halftrue_false": 3}
summary["_r"] = summary["scheme"].map(_scheme_rank)
summary = summary.sort_values(["_r", "text_mode", "macro_f1_mean"],
                              ascending=[True, True, False]).drop(columns="_r")

print("=" * 90)
print("PER-MODEL SUMMARY  (macro-F1 mean ± std across seeds)")
print("=" * 90)
print(summary.to_string(index=False))
summary.to_csv(f"{OUTPUT_DIR}/collapse_summary_by_model.csv", index=False)

In [ ]:
# ---- Headline effect-size table: DA-MiniLM-RWAC minus MiniLM-LogReg-RW ----
def effect_table(text_mode):
    rows = []
    for scheme in SCHEMES:
        K = COLLAPSE_SCHEMES[scheme]["num_classes"]
        sub = summary[(summary.scheme == scheme) & (summary.text_mode == text_mode)]
        try:
            f_rwac = sub.loc[sub.model == "DA-MiniLM-RWAC", "macro_f1_mean"].iloc[0]
            f_rw   = sub.loc[sub.model == "MiniLM-LogReg-RW", "macro_f1_mean"].iloc[0]
        except IndexError:
            continue
        abs_diff = f_rwac - f_rw
        rel = 100.0 * abs_diff / f_rw if f_rw else np.nan

        ci_l = ci_u = p = np.nan
        sig = None
        if len(bootstrap_results):
            b = bootstrap_results[(bootstrap_results.scheme == scheme) &
                                  (bootstrap_results.text_mode == text_mode)]
            if len(b):
                ci_l = b["ci_lower_95"].mean()
                ci_u = b["ci_upper_95"].mean()
                p    = b["p_value"].mean()
                sig  = bool((b["significant_at_95"]).mean() >= 0.5)
        rows.append({
            "scheme": scheme, "K": K,
            "RWAC_macroF1": round(f_rwac, 4), "LogRegRW_macroF1": round(f_rw, 4),
            "abs_diff": round(abs_diff, 4),
            "rel_pct": round(rel, 2) if rel == rel else np.nan,
            "boot_CI95_low": round(ci_l, 4) if ci_l == ci_l else np.nan,
            "boot_CI95_high": round(ci_u, 4) if ci_u == ci_u else np.nan,
            "boot_p": round(p, 4) if p == p else np.nan,
            "sig_at_95": sig,
        })
    return pd.DataFrame(rows)

for _mode in TEXT_MODES:
    print("=" * 90)
    print(f"HEADLINE EFFECT SIZE  DA-MiniLM-RWAC  minus  MiniLM-LogReg-RW   [text_mode={_mode}]")
    print("(absolute macro-F1 difference and 95% bootstrap CI reported before significance)")
    print("=" * 90)
    et = effect_table(_mode)
    print(et.to_string(index=False))
    et.to_csv(f"{OUTPUT_DIR}/collapse_effect_size_{_mode}.csv", index=False)
    print()

In [ ]:
from scipy import stats

def seed_level_effect(main_results, model_a="DA-MiniLM-RWAC", model_b="MiniLM-LogReg-RW",
                      text_mode=TEXT_MODE, split_mode=SPLIT_MODE):
    rows = []
    for scheme in SCHEMES:
        sub = main_results[(main_results.scheme == scheme) &
                           (main_results.text_mode == text_mode) &
                           (main_results.split_mode == split_mode)]
        a = sub[sub.model == model_a].sort_values("seed")["macro_f1"].values
        b = sub[sub.model == model_b].sort_values("seed")["macro_f1"].values
        if len(a) == 0 or len(b) == 0:
            continue
        # pair by position; if b is constant this is one-sample-on-diffs
        n = min(len(a), len(b))
        diffs = a[:n] - b[:n]
        mean = diffs.mean(); sd = diffs.std(ddof=1) if n > 1 else 0.0
        se = sd / np.sqrt(n) if n > 1 else 0.0
        tcrit = stats.t.ppf(0.975, n - 1) if n > 1 else 0.0
        ci_l, ci_u = mean - tcrit * se, mean + tcrit * se
        t_stat, p_t = stats.ttest_1samp(diffs, 0.0) if n > 1 else (np.nan, np.nan)
        n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum())
        p_sign = stats.binomtest(n_pos, n, 0.5).pvalue
        rows.append({
            "scheme": scheme, "K": COLLAPSE_SCHEMES[scheme]["num_classes"],
            "n_seeds": n, "mean_diff": round(mean, 4),
            "between_seed_sd": round(sd, 4),
            "t_CI95_low": round(ci_l, 4), "t_CI95_high": round(ci_u, 4),
            "t_p": round(p_t, 4) if p_t == p_t else np.nan,
            "seeds_favoring_A": f"{n_pos}/{n}",
            "sign_p": round(p_sign, 4),
        })
    df = pd.DataFrame(rows)
    _r = {"6class": 0, "3class": 1, "2class": 2, "2class_halftrue_false": 3}
    return df.assign(_r=df["scheme"].map(_r)).sort_values("_r").drop(columns="_r")

seed_effect = seed_level_effect(main_results)
print("=" * 96)
print(f"SEED-LEVEL EFFECT SIZE  DA-MiniLM-RWAC minus MiniLM-LogReg-RW  [text_mode={TEXT_MODE}]")
print("(between-seed variability; effect size + t-interval lead, sign test corroborates)")
print("=" * 96)
print(seed_effect.to_string(index=False))
seed_effect.to_csv(f"{OUTPUT_DIR}/collapse_seed_level_effect.csv", index=False)

print("""
Reading guide
-------------
- mean_diff + t_CI95: primary. If the interval excludes 0, the advantage is
  detectable at that granularity across seeds.
- seeds_favoring_A: corroboration. 5/5 is the strongest the sign test can show
  at n=5 (its p-value floor is 0.0625, so it never reaches 0.05 here by design).
- Compare against the Section 6 within-test bootstrap: agreement between the two
  (between-seed and within-test) is the robust claim.
""")

## Section 7 — Per-class recall at each granularity

The 6-class minority-collapse finding (rising accuracy alongside collapsing minority
recall) may change or disappear under label collapse — which is itself a result worth
reporting. This computes per-class recall for each model/scheme, averaged over seeds,
directly from stored predictions.

In [ ]:
from sklearn.metrics import recall_score

def per_class_recall_rows():
    rows = []
    for scheme in SCHEMES:
        order = COLLAPSE_SCHEMES[scheme]["order"]
        store = prediction_stores[scheme]
        for (model_name, text_mode, split_mode, seed), rec in store.items():
            if text_mode not in TEXT_MODES:
                continue
            y_true = rec["y_test"]; y_pred = rec["pred"]
            r = recall_score(y_true, y_pred, average=None,
                             labels=list(range(len(order))), zero_division=0)
            for cls_id, cls_name in enumerate(order):
                rows.append({"scheme": scheme, "model": model_name,
                             "text_mode": text_mode, "seed": seed,
                             "class": cls_name, "recall": r[cls_id]})
    return pd.DataFrame(rows)

pcr = per_class_recall_rows()
pcr_summary = (pcr.groupby(["scheme", "model", "text_mode", "class"])
                  .agg(recall_mean=("recall", "mean"), recall_std=("recall", "std"))
                  .reset_index())

for _mode in TEXT_MODES:
    print("=" * 90)
    print(f"PER-CLASS RECALL (mean over seeds)   [text_mode={_mode}]")
    print("=" * 90)
    piv = (pcr_summary[pcr_summary.text_mode == _mode]
           .pivot_table(index=["scheme", "model"], columns="class",
                        values="recall_mean"))
    print(piv.to_string())
    print()

pcr_summary.to_csv(f"{OUTPUT_DIR}/collapse_per_class_recall.csv", index=False)

In [ ]:
from sklearn.metrics import f1_score as _f1

def compute_chance_floors(df_labels_by_scheme, n_sim=2000, seed=0):
    """df_labels_by_scheme: {scheme -> 1D np.array of collapsed test-set label_ids}.
    Returns a tidy DataFrame of uniform / stratified / majority floors per scheme."""
    rows = []
    for scheme, y in df_labels_by_scheme.items():
        y = np.asarray(y).astype(int)
        K = int(y.max()) + 1
        counts = np.bincount(y, minlength=K).astype(float)
        p = counts / counts.sum()
        rng = np.random.RandomState(seed)

        # majority-class
        maj = np.full(len(y), int(p.argmax()))
        maj_acc = float((maj == y).mean())
        maj_f1 = _f1(y, maj, average="macro", zero_division=0)

        # Monte-Carlo stratified & uniform macro-F1
        strat_f1 = np.empty(n_sim); unif_f1 = np.empty(n_sim)
        for b in range(n_sim):
            strat_f1[b] = _f1(y, rng.choice(K, size=len(y), p=p), average="macro", zero_division=0)
            unif_f1[b]  = _f1(y, rng.randint(0, K, size=len(y)),  average="macro", zero_division=0)

        rows.append({
            "scheme": scheme, "K": K, "N_test": int(counts.sum()),
            "majority_prevalence": round(p.max(), 4),
            "uniform_acc_1overK": round(1.0 / K, 4),
            "uniform_macroF1": round(unif_f1.mean(), 4),
            "stratified_acc": round(float((p**2).sum()), 4),
            "stratified_macroF1": round(strat_f1.mean(), 4),
            "stratified_macroF1_p95": round(np.percentile(strat_f1, 95), 4),
            "majority_acc": round(maj_acc, 4),
            "majority_macroF1": round(maj_f1, 4),
        })
    return pd.DataFrame(rows)


# Build the TEST-set collapsed labels per scheme, using the SAME chronological split
# the battery used (seed 0 test indices; the test partition is identical across seeds
# for the chronological split, so seed choice does not matter here).
_splitter_ref = TemporalSplitter(train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15)
_test_labels_by_scheme = {}
for scheme in SCHEMES:
    df_c = LabelCollapser(scheme).transform(df6).reset_index(drop=True)
    _tr, _va, _te = _splitter_ref.get_split(df_c, SPLIT_MODE, seed=SEEDS[0])
    _test_labels_by_scheme[scheme] = df_c["label_id"].values[_te]

floors = compute_chance_floors(_test_labels_by_scheme, n_sim=2000, seed=0)

# Attach each model's observed macro-F1 and the margin over the stratified floor.
_scheme_rank = {"6class": 0, "3class": 1, "2class": 2, "2class_halftrue_false": 3}
obs = (main_results.groupby(["scheme", "model"])["macro_f1"].mean().reset_index())
margin_rows = []
for _, fr in floors.iterrows():
    s = fr["scheme"]; floor = fr["stratified_macroF1"]
    for model in ["DA-MiniLM-RWAC", "MiniLM-LogReg-RW", "Uniform-LR"]:
        m = obs[(obs.scheme == s) & (obs.model == model)]
        if len(m):
            mf = float(m["macro_f1"].iloc[0])
            margin_rows.append({"scheme": s, "K": int(fr["K"]), "model": model,
                                "macro_f1": round(mf, 4),
                                "stratified_floor": floor,
                                "margin_over_chance": round(mf - floor, 4)})
margin_df = pd.DataFrame(margin_rows)
margin_df["_r"] = margin_df["scheme"].map(_scheme_rank)
margin_df = margin_df.sort_values(["_r", "macro_f1"], ascending=[True, False]).drop(columns="_r")

pd.set_option("display.width", 200); pd.set_option("display.max_columns", None)
print("=" * 96)
print("CHANCE-LEVEL FLOORS PER GRANULARITY (macro-F1 chance is NOT 1/K)")
print("=" * 96)
floors["_r"] = floors["scheme"].map(_scheme_rank)
print(floors.sort_values("_r").drop(columns="_r").to_string(index=False))

print("\n" + "=" * 96)
print("MARGIN OVER CHANCE  (the honest cross-K quantity; raw macro-F1 is NOT comparable across K)")
print("=" * 96)
print(margin_df.to_string(index=False))

floors.drop(columns="_r").to_csv(f"{OUTPUT_DIR}/collapse_chance_floors.csv", index=False)
margin_df.to_csv(f"{OUTPUT_DIR}/collapse_margin_over_chance.csv", index=False)

## Section 8 — Saved artifacts

In [ ]:
import glob
print("Files written to", OUTPUT_DIR, ":")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/**/*.csv", recursive=True)):
    print("  ", f)

print("""
Artifact guide
--------------
collapse_main_results.csv        Every (scheme x model x seed) run, full metric row.
collapse_summary_by_model.csv    Macro-F1 mean +/- std per scheme/model (Section 6).
collapse_effect_size_*.csv       DA-MiniLM-RWAC minus MiniLM-LogReg-RW, abs diff +
                                 95% bootstrap CI per granularity (the headline table).
collapse_bootstrap_results.csv   Per-seed paired-bootstrap detail.
collapse_per_class_recall.csv    Per-class recall per scheme/model (minority-collapse
                                 diagnostic).
""")

## Section 9 — Reporting notes for the manuscript

- **Frame the collapse as an ablation of label granularity**, not a new task. The
  identical protocol, rows, embeddings, and comparators are held fixed; only the label
  resolution changes. The comparison to the 6-class result is therefore paired and
  clean.
- **Lead every claim with macro-F1 and the absolute effect size + CI**, then the
  significance label — matching the rest of the paper.
- **The 2-class boundary is a documented judgment call.** Report `2class`
  (half-true→true, LIAR convention) as primary and `2class_halftrue_false` as a
  robustness check; note whether the RWAC advantage's sign/magnitude is stable across
  the two boundaries.
- **The 3-class `{false, mixed, true}` result is directly comparable to MultiFC**, which
  is collapsed to the same three classes. If the LIAR2 3-class RWAC advantage behaves
  like the MultiFC 3-class result (advantage not reproducing / simpler comparator
  competitive), that *strengthens* the central claim rather than weakening it.
- **Report per-class recall under collapse**, especially whether the 6-class
  minority-class recall collapse (e.g. the "true" class) persists, softens, or vanishes
  once classes are merged. State this explicitly rather than only reporting aggregate
  macro-F1.
- **Calibration:** the two-stage (bug-fixed) scheme is used here. If any 6-class numbers
  in the paper were produced under the legacy `joint_macro_f1` scheme, note the
  calibration scheme alongside each granularity so the comparison is apples-to-apples.